In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if(torch.cuda.is_available()):
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'sample_data']


In [6]:
#upload dataset
from google.colab import files
uploaded=files.upload()

Saving Brain_tumar_dataset.zip to Brain_tumar_dataset.zip


In [7]:
#extract the database
import zipfile
import os

zip_path="/content/Brain_tumar_dataset.zip"
extract_path="./content/brain_tumor_data"

with zipfile.ZipFile(zip_path,"r") as zip_ref:
  zip_ref.extractall(extract_path)

print("Dataset Extracted")
print(os.listdir(extract_path))

Dataset Extracted
['Testing', 'Training']


In [13]:
#get folder strucutre
import os

for root,dir,files in os.walk("./content/brain_tumor_data",""):
  level=root.replace("./content/brain_tumor_data","").count(os.sep)

  if level <=2:
    print(" " * level + os.path.basename(root)+"/")

  glioma/
  meningioma/
  pituitary/
  notumor/
 Testing/
  glioma/
  meningioma/
  pituitary/
  notumor/
 Training/
brain_tumor_data/


In [14]:
#create data loaders
import torch
from torch.utils.data import DataLoader,random_split
from torchvision import datasets,transforms

#Reproducibility
SEED=42
torch.manual_seed(SEED)

#setting
IMAGE_SIZE=224
BATCH_SIZE=32

#path
train_dir="./content/brain_tumor_data/Training"
test_dir="./content/brain_tumor_data/Testing"

#imageNet normalization
mean=[0.485,0.456,0.406]
std=[0.229,0.224,0.225]

#training transformations
train_transform=transforms.Compose([
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean,std)
])


#validation/Test transformation
eval_transform=transforms.Compose([
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.transforms.Normalize(mean,std)
])

#load complete training data
full_dataset=datasets.ImageFolder(
    train_dir,
    transform=train_transform
)

#load test dataset
test_dataset=datasets.ImageFolder(
    test_dir,
    transform=eval_transform
)

#data split
train_size=int(0.8*len(full_dataset))
val_size=len(full_dataset)-train_size

train_dataset,val_dataset=random_split(
    full_dataset,
    [train_size,val_size],
    generator=torch.Generator().manual_seed(SEED)

)

#dataloader
train_loader=DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader=DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader=DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


print("Classes:", full_dataset.classes)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))


Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Training samples: 4480
Validation samples: 1120
Test samples: 1600
